# Sportwetten Vorhersage - Beispiel Analyse

Dieses Notebook demonstriert die Verwendung des Sportwetten-Vorhersage Systems.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.collect_data import SportsDataCollector
from src.data.preprocessing import DataPreprocessor
from src.models.train_model import SportsPredictor
from src.models.predict import MatchPredictor

sns.set_style('whitegrid')
%matplotlib inline

## 1. Daten laden

In [ ]:
# Lade Spieldaten
collector = SportsDataCollector()
matches_df = collector.load_data('matches.csv')
team_stats_df = collector.load_data('team_stats.csv')

print(f"Anzahl Spiele: {len(matches_df)}")
print(f"Anzahl Teams: {len(team_stats_df)}")

matches_df.head()

## 2. Explorative Datenanalyse

In [ ]:
# Verteilung der Ergebnisse
plt.figure(figsize=(10, 6))
matches_df['result'].value_counts().plot(kind='bar')
plt.title('Verteilung der Spielergebnisse')
plt.xlabel('Ergebnis')
plt.ylabel('Anzahl')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Top Teams nach Siegquote
plt.figure(figsize=(12, 6))
top_teams = team_stats_df.nlargest(10, 'win_rate')
sns.barplot(data=top_teams, x='team', y='win_rate')
plt.title('Top 10 Teams nach Siegquote')
plt.xlabel('Team')
plt.ylabel('Siegquote')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Modell laden und Vorhersagen machen

In [ ]:
# Lade trainiertes Modell
predictor = MatchPredictor(model_type='random_forest')

# Beispiel-Vorhersage
home_team = "Bayern München"
away_team = "Borussia Dortmund"

result, probabilities = predictor.predict_match(home_team, away_team)

print(f"\nSpiel: {home_team} vs {away_team}")
print(f"Vorhersage: {result}")
print(f"\nWahrscheinlichkeiten:")
print(f"  Heimsieg: {probabilities[0]*100:.2f}%")
print(f"  Unentschieden: {probabilities[1]*100:.2f}%")
print(f"  Auswärtssieg: {probabilities[2]*100:.2f}%")

In [ ]:
# Visualisiere Wahrscheinlichkeiten
plt.figure(figsize=(8, 6))
labels = ['Heimsieg', 'Unentschieden', 'Auswärtssieg']
plt.bar(labels, probabilities)
plt.title(f'{home_team} vs {away_team}')
plt.ylabel('Wahrscheinlichkeit')
plt.ylim([0, 1])
for i, prob in enumerate(probabilities):
    plt.text(i, prob + 0.02, f'{prob*100:.1f}%', ha='center')
plt.show()

## 4. Mehrere Spiele vorhersagen

In [ ]:
# Beispiel-Spiele
upcoming_matches = [
    ("Bayern München", "Borussia Dortmund"),
    ("RB Leipzig", "Bayer Leverkusen"),
    ("Union Berlin", "SC Freiburg"),
    ("Eintracht Frankfurt", "VfL Wolfsburg"),
]

predictions_list = []

for home, away in upcoming_matches:
    result, probs = predictor.predict_match(home, away)
    predictions_list.append({
        'Heimteam': home,
        'Auswärtsteam': away,
        'Vorhersage': result,
        'Heimsieg %': f"{probs[0]*100:.1f}",
        'Unentschieden %': f"{probs[1]*100:.1f}",
        'Auswärtssieg %': f"{probs[2]*100:.1f}",
        'Konfidenz': f"{max(probs)*100:.1f}"
    })

predictions_df = pd.DataFrame(predictions_list)
predictions_df